# Multimodal AML — Week 2: GraphSAGE Encoder

**Member A (Jaya) · DePaul SE 489 MLOps · 2026**

This notebook trains the **GraphSAGE graph encoder** on the Elliptic Bitcoin transaction graph.

### Architecture (matches proposal)
```
Input  : (N, 165)  node features
SAGE 1 : 165 → 128  ReLU + Dropout(0.3)
SAGE 2 : 128 → 64
Output : 64-dim embedding  ← fed to fusion head (Week 3)
Head   : 64 → 1, sigmoid   ← standalone eval
```

### Inputs (preprocessed by `elliptic_preprocess.py`)
```
data/processed/graph_features.npy    (46564, 165)
data/processed/graph_edge_index.npy  (2, 36624)
data/processed/graph_labels.npy      (46564,)   0=licit 1=illicit
data/processed/graph_node_ids.npy    (46564,)
```

### Outputs
```
models/graphsage/graphsage_best.pt      full model weights
models/graphsage/graphsage_encoder.pt   encoder only (for fusion)
graphsage_metrics.json                  val + test metrics
```

**Baseline to beat:** XGBoost AUC-PR = **0.9891** (Week 1)

## 1. Setup

In [ ]:
# Install PyTorch Geometric and MLflow (Colab already has PyTorch)
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')

!pip install -q torch-geometric mlflow
print('All packages installed.')

In [ ]:
import os, json, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    roc_auc_score, classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split

import mlflow

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('All imports OK')

## 2. Mount Google Drive & Load Data

The preprocessed `.npy` files live in the shared **MLOPS-data** Drive folder.  
Mount Drive, then either:
- **Option A** — pull via DVC (if you have credentials set up), or  
- **Option B** — copy files from the shared folder directly (simpler).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo to get code + DVC pointer files
!git clone -q https://github.com/jpmartin22/Multimodal_Anti_Money_Laundering /content/aml
%cd /content/aml
print('Repo cloned.')

In [ ]:
# Install DVC with Google Drive support
!pip install -q 'dvc[gdrive]'

# Configure OAuth credentials — get these from your Google Cloud Console
# (APIs & Services -> Credentials -> your Desktop OAuth client)
!dvc remote modify gdrive gdrive_client_id     'YOUR_GDRIVE_CLIENT_ID'
!dvc remote modify gdrive gdrive_client_secret 'YOUR_GDRIVE_CLIENT_SECRET'

# Pull only the graph processed files (skip large raw datasets)
!dvc pull data/processed/graph_features.npy \
           data/processed/graph_edge_index.npy \
           data/processed/graph_labels.npy \
           data/processed/graph_node_ids.npy

# Verify
for f in ['data/processed/graph_features.npy',
          'data/processed/graph_edge_index.npy',
          'data/processed/graph_labels.npy',
          'data/processed/graph_node_ids.npy']:
    status = '✅' if os.path.exists(f) else '❌ NOT FOUND'
    print(f'{status}  {f}')

## 3. Load & Inspect Graph Data

In [ ]:
features   = np.load('data/processed/graph_features.npy').astype(np.float32)
edge_index = np.load('data/processed/graph_edge_index.npy')
labels     = np.load('data/processed/graph_labels.npy')
node_ids   = np.load('data/processed/graph_node_ids.npy')

n_nodes     = features.shape[0]
n_features  = features.shape[1]
n_edges     = edge_index.shape[1]
n_illicit   = (labels == 1).sum()
n_licit     = (labels == 0).sum()

print('=' * 50)
print('Graph summary')
print('=' * 50)
print(f'Nodes          : {n_nodes:,}')
print(f'Node features  : {n_features}')
print(f'Edges          : {n_edges:,}')
print(f'  Illicit (1)  : {n_illicit:,}  ({100*n_illicit/n_nodes:.2f}%)')
print(f'  Licit   (0)  : {n_licit:,}  ({100*n_licit/n_nodes:.2f}%)')
print(f'Avg degree     : {n_edges/n_nodes:.2f}')
print(f'Class imbalance: {n_licit/n_illicit:.1f}:1')
print('=' * 50)

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Class distribution bar
axes[0].bar(['Licit (0)', 'Illicit (1)'], [n_licit, n_illicit],
            color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.2)
for i, (name, val) in enumerate(zip(['Licit', 'Illicit'], [n_licit, n_illicit])):
    axes[0].text(i, val + 300, f'{val:,}\n({100*val/n_nodes:.1f}%)',
                 ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Node Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Degree distribution
out_deg = np.bincount(edge_index[0], minlength=n_nodes)
in_deg  = np.bincount(edge_index[1], minlength=n_nodes)
total_deg = out_deg + in_deg

axes[1].hist(total_deg[labels==0].clip(max=30), bins=30, alpha=0.6,
             color='#2ecc71', label='Licit', density=True)
axes[1].hist(total_deg[labels==1].clip(max=30), bins=30, alpha=0.6,
             color='#e74c3c', label='Illicit', density=True)
axes[1].set_title('Degree Distribution by Class (capped at 30)', fontweight='bold')
axes[1].set_xlabel('Total degree')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('Elliptic Graph — EDA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('graph_eda.png', bbox_inches='tight')
plt.show()

print(f'Mean degree — Licit  : {total_deg[labels==0].mean():.2f}')
print(f'Mean degree — Illicit: {total_deg[labels==1].mean():.2f}')

In [ ]:
# Top-20 most discriminative features (abs mean difference)
mean_illicit = features[labels == 1].mean(axis=0)
mean_licit   = features[labels == 0].mean(axis=0)
diff = np.abs(mean_illicit - mean_licit)
top20 = np.argsort(diff)[::-1][:20]

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(20)
w = 0.35
ax.bar(x - w/2, mean_illicit[top20], w, label='Illicit', color='#e74c3c', alpha=0.8)
ax.bar(x + w/2, mean_licit[top20],   w, label='Licit',   color='#2ecc71', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f'f{i}' for i in top20], rotation=45, ha='right')
ax.set_title('Top-20 Features by |Illicit – Licit| Mean Difference', fontweight='bold')
ax.set_ylabel('Mean value')
ax.legend()
plt.tight_layout()
plt.savefig('feature_separability.png', bbox_inches='tight')
plt.show()

## 5. GraphSAGE Model

In [ ]:
class GraphSAGEEncoder(nn.Module):
    """
    2-layer GraphSAGE encoder.
    Input  : (N, 165) + edge_index
    Output : (N, 64)  node embeddings  ← fusion head input
    """
    def __init__(self, in_channels=165, hidden_channels=128,
                 embedding_dim=64, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, embedding_dim)
        self.dropout = dropout
        self.embedding_dim = embedding_dim

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x


class GraphSAGEClassifier(nn.Module):
    """Encoder + binary head. Encoder saved separately for fusion."""
    def __init__(self, in_channels=165, hidden_channels=128,
                 embedding_dim=64, dropout=0.3):
        super().__init__()
        self.encoder = GraphSAGEEncoder(
            in_channels, hidden_channels, embedding_dim, dropout
        )
        self.head = nn.Linear(embedding_dim, 1)

    def forward(self, x, edge_index):
        emb = self.encoder(x, edge_index)
        logit = self.head(emb).squeeze(-1)
        return logit, emb


# Quick architecture check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

_m = GraphSAGEClassifier()
print(_m)

## 6. Data Preparation

In [ ]:
# Sanity checks
assert features.ndim == 2
assert labels.ndim == 1
assert features.shape[0] == labels.shape[0]
assert edge_index.shape[0] == 2
assert np.isnan(features).sum() == 0, 'NaN in features!'
assert set(np.unique(labels)).issubset({0, 1})
print('All sanity checks passed ✅')

# Stratified 70 / 15 / 15 split
idx = np.arange(n_nodes)
idx_trainval, idx_test = train_test_split(
    idx, test_size=0.15, stratify=labels, random_state=SEED
)
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=0.15/0.85,
    stratify=labels[idx_trainval], random_state=SEED
)

train_mask = torch.zeros(n_nodes, dtype=torch.bool)
val_mask   = torch.zeros(n_nodes, dtype=torch.bool)
test_mask  = torch.zeros(n_nodes, dtype=torch.bool)
train_mask[idx_train] = True
val_mask[idx_val]     = True
test_mask[idx_test]   = True

print(f'Train: {train_mask.sum():,} | Val: {val_mask.sum():,} | Test: {test_mask.sum():,}')

# Build PyG Data object
data = Data(
    x          = torch.tensor(features, dtype=torch.float32),
    edge_index = torch.tensor(edge_index, dtype=torch.long),
    y          = torch.tensor(labels, dtype=torch.float32),
).to(device)

train_mask = train_mask.to(device)
val_mask   = val_mask.to(device)
test_mask  = test_mask.to(device)

# Class weight for imbalanced training
y_train  = labels[idx_train]
n_neg, n_pos = (y_train == 0).sum(), (y_train == 1).sum()
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(device)
print(f'pos_weight: {pos_weight.item():.2f}x  (neg={n_neg:,} pos={n_pos:,})')

## 7. Train GraphSAGE

In [ ]:
def evaluate(model, data, mask):
    model.eval()
    with torch.no_grad():
        logits, _ = model(data.x, data.edge_index)
        probs  = torch.sigmoid(logits[mask]).cpu().numpy()
        labels_np = data.y[mask].cpu().numpy()

    auc_pr = average_precision_score(labels_np, probs)

    precision, recall, thresholds = precision_recall_curve(labels_np, probs)
    f1_scores  = 2 * precision * recall / (precision + recall + 1e-8)
    best_idx   = np.argmax(f1_scores[:-1])
    best_thresh = float(thresholds[best_idx])
    preds = (probs >= best_thresh).astype(int)

    idx_r80 = np.searchsorted(recall[::-1], 0.8)
    prec_at_r80 = float(precision[::-1][idx_r80]) if idx_r80 < len(precision) else 0.0

    report = classification_report(labels_np, preds, output_dict=True, zero_division=0)
    return {
        'auc_pr'           : round(float(auc_pr), 4),
        'prec_at_recall_80': round(prec_at_r80, 4),
        'f1_fraud'         : round(report.get('1', {}).get('f1-score', 0), 4),
        'recall_fraud'     : round(report.get('1', {}).get('recall', 0), 4),
        'precision_fraud'  : round(report.get('1', {}).get('precision', 0), 4),
        'false_positive_rate': round(1 - report.get('0', {}).get('recall', 1.0), 4),
        'accuracy'         : round(report.get('accuracy', 0), 4),
        'optimal_threshold': round(best_thresh, 4),
    }

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
HIDDEN_DIM  = 128
DROPOUT     = 0.3
LR          = 0.001
EPOCHS      = 200
EVAL_EVERY  = 20    # evaluate on val set every N epochs

print(f'hidden_dim={HIDDEN_DIM} | dropout={DROPOUT} | lr={LR} | epochs={EPOCHS}')

In [ ]:
os.makedirs('models/graphsage', exist_ok=True)

model     = GraphSAGEClassifier(
    in_channels=n_features, hidden_channels=HIDDEN_DIM,
    embedding_dim=64, dropout=DROPOUT
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=5, factor=0.5
)

mlflow.set_experiment('aml_graphsage_graph')

history = {'epoch': [], 'train_loss': [], 'val_auc_pr': [], 'val_f1': []}

best_val_auc = 0.0
best_epoch   = 0
train_start  = time.time()

with mlflow.start_run(run_name=f'graphsage_h{HIDDEN_DIM}_lr{LR}_ep{EPOCHS}'):
    mlflow.log_params({
        'lr': LR, 'hidden_dim': HIDDEN_DIM, 'embedding_dim': 64,
        'dropout': DROPOUT, 'epochs': EPOCHS,
        'n_nodes': n_nodes, 'n_edges': n_edges,
        'n_features': n_features,
        'pos_weight': round(float(pos_weight.item()), 2),
    })

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        logits, _ = model(data.x, data.edge_index)
        loss = criterion(logits[train_mask], data.y[train_mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if epoch % EVAL_EVERY == 0 or epoch == 1:
            val_m = evaluate(model, data, val_mask)
            scheduler.step(val_m['auc_pr'])

            history['epoch'].append(epoch)
            history['train_loss'].append(round(loss.item(), 4))
            history['val_auc_pr'].append(val_m['auc_pr'])
            history['val_f1'].append(val_m['f1_fraud'])

            mlflow.log_metrics({
                'train_loss'   : round(loss.item(), 4),
                'val_auc_pr'   : val_m['auc_pr'],
                'val_f1_fraud' : val_m['f1_fraud'],
                'val_recall'   : val_m['recall_fraud'],
            }, step=epoch)

            print(f'Epoch {epoch:3d}/{EPOCHS} | loss: {loss.item():.4f} | '
                  f'val AUC-PR: {val_m["auc_pr"]:.4f} | '
                  f'val F1: {val_m["f1_fraud"]:.4f}')

            if val_m['auc_pr'] > best_val_auc:
                best_val_auc = val_m['auc_pr']
                best_epoch   = epoch
                torch.save(model.state_dict(),
                           'models/graphsage/graphsage_best.pt')
                torch.save(model.encoder.state_dict(),
                           'models/graphsage/graphsage_encoder.pt')
                print(f'  → New best: {best_val_auc:.4f} — saved')

    # ── Final eval ────────────────────────────────────────────────────────────
    model.load_state_dict(
        torch.load('models/graphsage/graphsage_best.pt', map_location=device)
    )
    val_metrics  = evaluate(model, data, val_mask)
    test_metrics = evaluate(model, data, test_mask)
    total_min    = (time.time() - train_start) / 60

    mlflow.log_metrics({f'val_{k}' : v for k, v in val_metrics.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in test_metrics.items()})
    mlflow.log_metric('training_time_min', round(total_min, 2))
    mlflow.log_metric('best_epoch', best_epoch)
    mlflow.log_artifact('models/graphsage/graphsage_best.pt')
    mlflow.log_artifact('models/graphsage/graphsage_encoder.pt')

print(f'\nTraining complete in {total_min:.1f} min | Best epoch: {best_epoch}')

## 8. Evaluation

In [ ]:
import pandas as pd

targets = {
    'auc_pr'            : ('AUC-PR (primary)',      '≥ 0.80',  '>0.9891 (beat XGB)'),
    'prec_at_recall_80' : ('Prec @ Recall=0.80',   '≥ 0.70',  ''),
    'false_positive_rate': ('FPR @ Recall=0.80',   '≤ 0.05',  ''),
    'f1_fraud'          : ('F1 (fraud class)',      '—',       ''),
    'recall_fraud'      : ('Recall (fraud)',        '—',       ''),
    'accuracy'          : ('Accuracy',              '—',       ''),
}

rows = []
for key, (label, target, note) in targets.items():
    rows.append({
        'Metric'     : label,
        'Val'        : val_metrics.get(key, '—'),
        'Test'       : test_metrics.get(key, '—'),
        'Target'     : target,
        'Note'       : note,
    })

results_df = pd.DataFrame(rows).set_index('Metric')

print('=' * 70)
print('GraphSAGE Results')
print('=' * 70)
print(results_df.to_string())
print('=' * 70)
print(f'\nXGBoost baseline AUC-PR: 0.9891')
delta = test_metrics['auc_pr'] - 0.9891
sign  = '+' if delta >= 0 else ''
print(f'GraphSAGE test AUC-PR : {test_metrics["auc_pr"]:.4f}  ({sign}{delta:.4f} vs baseline)')

In [ ]:
# Precision-Recall curve + training curves
model.eval()
with torch.no_grad():
    logits, _ = model(data.x, data.edge_index)
    probs_test  = torch.sigmoid(logits[test_mask]).cpu().numpy()
    labels_test = data.y[test_mask].cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# PR curve
prec, rec, _ = precision_recall_curve(labels_test, probs_test)
auc_pr_test  = average_precision_score(labels_test, probs_test)
axes[0].plot(rec, prec, color='#2980b9', linewidth=2,
             label=f'GraphSAGE (AUC-PR = {auc_pr_test:.4f})')
axes[0].axhline(labels_test.mean(), color='gray', linestyle='--',
                label=f'Random ({labels_test.mean():.3f})')
axes[0].axvline(0.80, color='#e74c3c', linestyle=':', linewidth=1.5,
                label='Recall = 0.80 target')
# XGB baseline reference
axes[0].axhline(0.9891, color='orange', linestyle='--', linewidth=1,
                label='XGB baseline AUC-PR')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve (Test)', fontweight='bold')
axes[0].legend(fontsize=8); axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1.05])

# Training loss
axes[1].plot(history['epoch'], history['train_loss'],
             color='#e74c3c', linewidth=2, marker='o', markersize=4)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('BCE Loss')
axes[1].set_title('Training Loss', fontweight='bold')

# Val AUC-PR
axes[2].plot(history['epoch'], history['val_auc_pr'],
             color='#2980b9', linewidth=2, marker='o', markersize=4, label='Val AUC-PR')
axes[2].axhline(0.9891, color='orange', linestyle='--', linewidth=1,
                label='XGB baseline')
axes[2].axvline(best_epoch, color='green', linestyle=':', linewidth=1.5,
                label=f'Best epoch ({best_epoch})')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('AUC-PR')
axes[2].set_title('Validation AUC-PR', fontweight='bold')
axes[2].legend(fontsize=8)

plt.suptitle('GraphSAGE — Training & Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('graphsage_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix at optimal threshold
best_thresh = test_metrics['optimal_threshold']
preds = (probs_test >= best_thresh).astype(int)
cm = confusion_matrix(labels_test, preds)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
            xticklabels=['Pred Licit','Pred Illicit'],
            yticklabels=['True Licit','True Illicit'], ax=ax)
ax.set_title(f'Confusion Matrix @ threshold={best_thresh:.3f}', fontweight='bold')
plt.tight_layout()
plt.savefig('graphsage_confusion.png', bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Positives  (caught illicit): {tp}')
print(f'False Negatives (missed illicit): {fn}')
print(f'False Positives (false alarms)  : {fp}')
print(f'True Negatives                  : {tn}')

In [ ]:
# t-SNE of 64-dim embeddings (sample 2000 nodes)
from sklearn.manifold import TSNE

model.eval()
with torch.no_grad():
    _, embeddings = model(data.x, data.edge_index)
    embeddings = embeddings.cpu().numpy()

sample_idx = np.random.choice(n_nodes, size=2000, replace=False)
emb_sample = embeddings[sample_idx]
lbl_sample = labels[sample_idx]

print('Running t-SNE (may take ~1 min)...')
tsne = TSNE(n_components=2, random_state=SEED, perplexity=40)
emb_2d = tsne.fit_transform(emb_sample)

fig, ax = plt.subplots(figsize=(8, 6))
for label_val, color, name in [(0, '#2ecc71', 'Licit'), (1, '#e74c3c', 'Illicit')]:
    mask = lbl_sample == label_val
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
               c=color, label=f'{name} ({mask.sum()})',
               s=8, alpha=0.6)
ax.set_title('t-SNE of GraphSAGE 64-dim Embeddings (2000 nodes)', fontweight='bold')
ax.legend(markerscale=3)
ax.axis('off')
plt.tight_layout()
plt.savefig('graphsage_tsne.png', bbox_inches='tight')
plt.show()

## 9. Save Results & Upload to Drive

In [ ]:
out = {
    'val_metrics' : val_metrics,
    'test_metrics': test_metrics,
    'best_epoch'  : best_epoch,
    'training_time_min': round(total_min, 2),
    'hyperparams' : {
        'lr': LR, 'hidden_dim': HIDDEN_DIM,
        'embedding_dim': 64, 'dropout': DROPOUT, 'epochs': EPOCHS,
    },
    'baseline_comparison': {
        'xgboost_test_auc_pr' : 0.9891,
        'graphsage_test_auc_pr': test_metrics['auc_pr'],
        'delta' : round(test_metrics['auc_pr'] - 0.9891, 4),
    }
}

with open('graphsage_metrics.json', 'w') as f:
    json.dump(out, f, indent=2)

print(json.dumps(out, indent=2))

In [ ]:
import shutil

# Copy models and metrics to your Drive so DVC can track them
DRIVE_OUT = '/content/drive/MyDrive/multimodal_anti_money_laundering'
os.makedirs(f'{DRIVE_OUT}/models/graphsage', exist_ok=True)

shutil.copy('models/graphsage/graphsage_best.pt',
            f'{DRIVE_OUT}/models/graphsage/graphsage_best.pt')
shutil.copy('models/graphsage/graphsage_encoder.pt',
            f'{DRIVE_OUT}/models/graphsage/graphsage_encoder.pt')
shutil.copy('graphsage_metrics.json',
            f'{DRIVE_OUT}/graphsage_metrics.json')

for img in ['graph_eda.png', 'feature_separability.png',
            'graphsage_curves.png', 'graphsage_confusion.png', 'graphsage_tsne.png']:
    if os.path.exists(img):
        shutil.copy(img, f'{DRIVE_OUT}/{img}')

print('All outputs saved to Google Drive ✅')
print(f'Path: {DRIVE_OUT}')

## 10. Summary

### GraphSAGE Architecture
| Layer | Dims | Activation |
|-------|------|------------|
| SAGEConv 1 | 165 → 128 | ReLU + Dropout(0.3) |
| SAGEConv 2 | 128 → 64  | — (embedding output) |
| Head       | 64  → 1   | Sigmoid |

### Results vs Baseline
| Metric | XGBoost (Week 1) | GraphSAGE (Week 2) | Target |
|--------|------------------|--------------------|--------|
| **AUC-PR** | 0.9891 | *see above* | ≥ 0.80 |
| Prec@Recall=0.80 | 1.0000 | *see above* | ≥ 0.70 |
| FPR@Recall=0.80 | 0.0000 | *see above* | ≤ 0.05 |

### Key Insight
GraphSAGE learns **inductive** node embeddings — it can generalize to unseen nodes at inference time, unlike the transductive XGBoost baseline. The 64-dim embeddings produced here are the graph modality input to the **Week 3 fusion head**.

### Next Step (Week 3 — Member A)
- Build **late-fusion MLP** combining GraphSAGE (64-dim) + BiLSTM (64-dim) + DistilBERT (64-dim) embeddings
- Add **Platt calibration** for well-calibrated fraud probabilities
- Add **SHAP force plots** for explainability